In [3]:
from pathlib import Path
import pandas as pd

# 1. Definir rutas desde el notebook (subiendo un nivel a la raíz)
root = Path().resolve().parent
raw_path = root / "data/raw/MunicipiosSequia.xlsx"
interim_dir = root / "data/interim"
interim_dir.mkdir(parents=True, exist_ok=True)
interim_path = interim_dir / "sonora_sequia_tidy.csv"

print("Leyendo el archivo crudo de CONAGUA...")
df = pd.read_excel(raw_path, engine="openpyxl")

# 2. Filtrar Sonora de forma segura (usando clave que empiece con '26')
cve_col = next((c for c in df.columns if "CVE" in str(c).upper()), None)
if cve_col:
  df_sonora = df[df[cve_col].astype(str).str.zfill(5).str.startswith("26")].copy()
elif "Entidad" in df.columns:
  df_sonora = df[df["Entidad"].str.lower() == "sonora"].copy()
else:
  df_sonora = df.copy()

# 3. Identificar columnas fijas (metadatos) vs columnas de fechas (quincenas)
id_vars_candidates = [
    "Cve_Entidad",
    "Entidad",
    "Cve_Municipio",
    "Municipio",
    "CVE_CONCATENADA",
    "CVE_ENT",
    "CVE_MUN",
    "MUNICIPIO",
    "Abrevia",
]
id_vars = [c for c in df_sonora.columns if c in id_vars_candidates]

# 4. Aplicar melt para pasar a formato largo (Tidy Data)
print("Transformando a formato largo (Tidy Data)...")
df_melted = df_sonora.melt(
    id_vars=id_vars, var_name="fecha", value_name="categoria_sequia"
)

# Limpiar y convertir fechas
df_melted["fecha"] = pd.to_datetime(df_melted["fecha"], errors="coerce")
df_melted = df_melted.dropna(subset=["fecha"])

# 5. Filtrar estrictamente el rango de años (2018 a 2025)
df_melted["Anio"] = df_melted["fecha"].dt.year
df_filtered = df_melted[
    (df_melted["Anio"] >= 2018) & (df_melted["Anio"] <= 2025)
].copy()

# 6. Añadir escala numérica de severidad (Ideal para correlaciones y gráficas en el EDA)
severity_map = {"Sin Sequía": 0, "D0": 1, "D1": 2, "D2": 3, "D3": 4, "D4": 5}
df_filtered["severidad_num"] = (
    df_filtered["categoria_sequia"].map(severity_map).fillna(0).astype(int)
)

# 7. Guardar en la capa interim
df_filtered.to_csv(interim_path, index=False, encoding="utf-8-sig")
print(f"¡Datos procesados y guardados con éxito en: {interim_path}!")
print(f"Dimensiones de la tabla final (Tidy Data): {df_filtered.shape}")

# Vista previa lista para EDA
display(df_filtered.head(3))

Leyendo el archivo crudo de CONAGUA...
Transformando a formato largo (Tidy Data)...


C:\Users\patyq\AppData\Local\Temp\ipykernel_22604\3251021111.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted["fecha"] = pd.to_datetime(df_melted["fecha"], errors="coerce")


¡Datos procesados y guardados con éxito en: C:\Users\patyq\Documents\sonora-agriclimate-eda\data\interim\sonora_sequia_tidy.csv!
Dimensiones de la tabla final (Tidy Data): (13752, 7)


,CVE_CONCATENADA,CVE_ENT,CVE_MUN,fecha,categoria_sequia,Anio,severidad_num
16776,26001,26,1,2018-01-15,D1,2018,2
16777,26002,26,2,2018-01-15,D1,2018,2
16778,26003,26,3,2018-01-15,D2,2018,3
